# FSDP / ZeRO Sharding from Scratch

Build order:
1. **Stage 1** — Memory anatomy: understand what takes memory during training
2. **Stage 2** — Simulate ZeRO Stages 1, 2, 3 mathematically (single process)
3. **Stage 3** — DDP baseline with `torch.multiprocessing.spawn`
4. **Stage 4** — FSDP with PyTorch's native API
5. **Stage 5** — Memory & communication analysis

**Key insight before starting:**  
DDP replicates the full model on every GPU and does an all-reduce on gradients.  
ZeRO eliminates this redundancy by sharding across `N` ranks:

| ZeRO Stage | What's sharded | Memory per GPU |
|---|---|---|
| 0 (DDP)   | nothing             | full model × 1 |
| 1         | optimizer states    | params + grads + optim/N |
| 2         | + gradients         | params + (grads + optim)/N |
| 3 (FSDP)  | + parameters        | (params + grads + optim)/N |

Model pair for experiments: a small GPT-style transformer you'll define from scratch.

## Setup

In [ ]:
!pip install torch --quiet

In [ ]:
import os
import sys
import math
import time
import copy
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass
from functools import partial
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

# Ensure we can import the fsdp_helpers module (which lives next to this notebook).
# mp.spawn uses the "spawn" start method, which pickles functions by module reference.
# Functions defined in notebook cells live in __main__ and cannot be found by child
# processes, so they must be defined in a proper .py module instead.
for _p in [os.getcwd(), os.path.join(os.getcwd(), "training"), ".", "training"]:
    if os.path.isfile(os.path.join(_p, "fsdp_helpers.py")):
        if _p not in sys.path:
            sys.path.insert(0, os.path.abspath(_p))
        break

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"GPU count:       {torch.cuda.device_count()}")

# If no GPUs, we still simulate the math — that's the bulk of the learning
WORLD_SIZE = max(torch.cuda.device_count(), 1)
USE_GPU    = torch.cuda.is_available()
print(f"\nSimulating {WORLD_SIZE} ranks  (GPU={'yes' if USE_GPU else 'no — CPU simulation'})")

### Toy model: small GPT-style transformer

We use this throughout so we can inspect real memory numbers.

In [ ]:
# Model classes are defined in fsdp_helpers.py so that mp.spawn workers can
# reference them.  See the module for the full source code.
from fsdp_helpers import ModelConfig, CausalSelfAttention, TransformerBlock, ToyGPT

cfg   = ModelConfig()
model = ToyGPT(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}  ({n_params/1e6:.1f}M)")

---
## Stage 1 — Memory Anatomy

Before sharding anything, understand what actually occupies GPU memory during training.
There are four buckets:

```
Parameters      — the weights themselves
Gradients       — same shape as parameters, built during backward
Optimizer state — Adam keeps momentum + variance per param (2× param count)
Activations     — intermediate tensors saved for backward (scales with batch & seq_len)
```

In [ ]:
def bytes_to_mb(b: int) -> float:
    return b / (1024 ** 2)

def count_param_bytes(model: nn.Module, dtype=torch.float32) -> int:
    """
    Total bytes occupied by model parameters.
    dtype determines bytes-per-element (fp32=4, fp16=2, bf16=2).
    """
    bytes_per_elem = torch.finfo(dtype).bits // 8
    return sum(p.numel() for p in model.parameters()) * bytes_per_elem

def estimate_adam_optimizer_bytes(model: nn.Module) -> int:
    """
    Adam stores: momentum (m) + variance (v) per parameter, both in fp32.
    This is 2 × param_count × 4 bytes regardless of param dtype.

    TODO: implement this.
    Hint: iterate model.parameters(), sum numel(), multiply by 2 × 4.
    """
    raise NotImplementedError

def estimate_gradient_bytes(model: nn.Module, dtype=torch.float32) -> int:
    """
    Gradients have the same shape as parameters.
    In mixed precision, gradients are kept in fp32 for numerical stability.

    TODO: implement this.
    """
    raise NotImplementedError

def estimate_activation_bytes(
    cfg: ModelConfig,
    batch_size: int,
    dtype=torch.float32,
) -> int:
    """
    Activation memory is harder to compute exactly — it depends on which
    intermediates PyTorch saves for the backward pass.

    A reasonable approximation for a transformer:
      Per layer: attention scores + residual stream + FF intermediates
      ≈ batch × seq_len × d_model × 34  (empirical constant for fp32)

    TODO: implement this approximation.
    Formula: n_layers × batch_size × seq_len × d_model × 34 × bytes_per_elem

    Reference: Reducing Activation Recomputation in Large Transformer Models
    (Korthikanti et al. 2022) derives the constant more rigorously.
    """
    raise NotImplementedError

def memory_breakdown(
    model: nn.Module,
    cfg: ModelConfig,
    batch_size: int = 4,
    param_dtype = torch.float32,
) -> dict:
    """
    Return a dict with MB estimates for each memory bucket.

    TODO: call the three functions above and assemble results.
    Include a 'total' key that sums params + grads + optimizer + activations.
    """
    raise NotImplementedError

In [ ]:
# Sanity check — does the breakdown make sense?
bd = memory_breakdown(model, cfg, batch_size=4)
for k, v in bd.items():
    print(f"{k:>12}: {v:>8.1f} MB")

# Rule of thumb: Adam fp32 training ≈ 16 bytes per parameter
#   params(4) + grads(4) + momentum(4) + variance(4) = 16
expected_no_activation = n_params * 16 / (1024**2)
print(f"\n16 bytes/param rule:  {expected_no_activation:.1f} MB  (should match params+grads+optimizer)")

In [ ]:
# Plot memory breakdown as a stacked bar
# TODO: write a plot_memory_breakdown(bd) function
# Each bucket gets its own color; show MB labels on each segment

def plot_memory_breakdown(bd: dict, title="Memory breakdown (1 GPU, no sharding)"):
    raise NotImplementedError

plot_memory_breakdown(bd)

---
## Stage 2 — Simulate ZeRO Stages Mathematically

We don't need multiple real GPUs to understand ZeRO — we can simulate the
memory math of N ranks on a single process.

ZeRO (Zero Redundancy Optimizer) has three stages, each sharding one more bucket:

```
Stage 1:  each rank holds 1/N of optimizer states
Stage 2:  each rank holds 1/N of optimizer states + 1/N of gradients
Stage 3:  each rank holds 1/N of everything (params + grads + optim)
```

Communication cost increases with each stage:
```
DDP:       all-reduce gradients        = 2Ψ   (Ψ = param bytes)
ZeRO-1:    reduce-scatter grads + all-gather optim update = 2Ψ
ZeRO-2:    reduce-scatter grads        = Ψ
ZeRO-3:    all-gather params (fwd+bwd) + reduce-scatter grads = 3Ψ
```

In [ ]:
def zero_memory_per_rank(
    n_params:     int,
    world_size:   int,
    zero_stage:   int,   # 0=DDP, 1, 2, 3
    param_dtype:  torch.dtype = torch.float32,
    batch_size:   int = 4,
    cfg:          ModelConfig = None,
) -> dict:
    """
    Compute memory per GPU for a given ZeRO stage.

    Memory rules:
      param bytes   = n_params × sizeof(param_dtype)
      grad bytes    = n_params × 4   (fp32 grads for stability)
      optim bytes   = n_params × 8   (Adam: momentum + variance, fp32)
      activation MB = estimate_activation_bytes(cfg, batch_size) if cfg given

    ZeRO sharding:
      stage 0: full params + full grads + full optim
      stage 1: full params + full grads + optim/N
      stage 2: full params + grads/N   + optim/N
      stage 3: params/N   + grads/N   + optim/N

    Note: stage 3 needs a temporary all-gather buffer during forward/backward
    to reconstruct full layers. Add params/N as overhead for that.

    TODO: implement this function.
    Return dict with keys: params_mb, grads_mb, optim_mb, activations_mb, total_mb
    """
    raise NotImplementedError

def compare_zero_stages(
    model:      nn.Module,
    cfg:        ModelConfig,
    world_sizes: list = [1, 2, 4, 8, 16, 64],
    batch_size: int = 4,
) -> dict:
    """
    For each (world_size, zero_stage) combination, compute memory per rank.

    Return nested dict: results[world_size][stage] = memory_dict

    TODO: implement by calling zero_memory_per_rank for stages 0-3
    across all world_sizes.
    """
    raise NotImplementedError

In [ ]:
results = compare_zero_stages(model, cfg)

# Print table: world_size × stage → total MB
world_sizes = [1, 2, 4, 8, 16, 64]
print(f"{'N':>4}  {'DDP(0)':>10}  {'ZeRO-1':>10}  {'ZeRO-2':>10}  {'ZeRO-3':>10}")
print("-" * 52)
for N in world_sizes:
    row = [f"{results[N][s]['total_mb']:>10.1f}" for s in [0, 1, 2, 3]]
    print(f"{N:>4}  {'  '.join(row)}")

In [ ]:
# Plot: memory per rank vs world_size for each ZeRO stage
# TODO: plot 4 lines (one per stage), x=world_size (log scale), y=memory MB
# Add a horizontal dashed line for a hypothetical GPU memory limit (e.g. 40GB)

def plot_zero_scaling(results: dict, gpu_memory_limit_gb: float = 40.0):
    """
    Plot memory per rank vs world size for each ZeRO stage.
    Dashed horizontal line shows GPU memory capacity.
    """
    raise NotImplementedError

plot_zero_scaling(results)

### 2b — Simulate the communication pattern

Implement reduce-scatter and all-gather on CPU tensors to feel
how ZeRO-3 / FSDP reconstructs parameters during forward pass.

In [ ]:
def simulate_all_reduce(tensors: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    Simulate an all-reduce across N ranks (DDP style).
    Each rank holds a full gradient tensor.
    After all-reduce, every rank has the sum (then divided by N for mean).

    Input:  tensors[i] = gradient on rank i  — all same shape
    Output: list of N tensors, each = mean of all inputs

    TODO: implement.
    Hint: sum all tensors, divide by N, return a list of N copies.
    Communication volume: 2 × total_elements × bytes  (send + receive)
    """
    raise NotImplementedError

def simulate_reduce_scatter(tensors: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    Simulate reduce-scatter across N ranks (ZeRO-2/3 style).
    Each rank holds a full gradient tensor.
    After reduce-scatter:
      rank i holds the reduced (summed) shard for positions [i*shard_size : (i+1)*shard_size]

    Input:  tensors[i] = full gradient on rank i  — all shape [total_params]
    Output: list of N tensors, each shape [total_params // N]

    TODO: implement.
    Hint:
      1. Sum all tensors element-wise
      2. Split the sum into N equal chunks
      3. Return the chunk list (rank i gets chunk i)
    Communication volume: total_elements × bytes  (half of all-reduce!)
    """
    raise NotImplementedError

def simulate_all_gather(shards: list[torch.Tensor]) -> list[torch.Tensor]:
    """
    Simulate all-gather across N ranks (ZeRO-3 / FSDP param reconstruction).
    Each rank holds a shard of the parameter tensor.
    After all-gather, every rank has the full reconstructed tensor.

    Input:  shards[i] = parameter shard on rank i  — shape [total_params // N]
    Output: list of N tensors, each = concatenation of all shards

    TODO: implement.
    Hint: concatenate all shards along dim 0, return N copies.
    Communication volume: total_elements × bytes
    """
    raise NotImplementedError

In [ ]:
# Verify communication primitives
N = 4
total_params = 1000

# Each rank has different gradients
grads = [torch.ones(total_params) * (i + 1) for i in range(N)]

# all-reduce: each rank should get mean = (1+2+3+4)/4 = 2.5
reduced = simulate_all_reduce(grads)
assert all(torch.allclose(r, torch.full((total_params,), 2.5)) for r in reduced), "all-reduce bug"
print("✓ all-reduce: each rank has mean gradient")

# reduce-scatter: rank i holds shard i of the sum
scattered = simulate_reduce_scatter(grads)
full_sum   = sum(g for g in grads)  # element-wise sum = 10.0 everywhere
assert len(scattered) == N
assert all(s.shape[0] == total_params // N for s in scattered), "shard size bug"
recon = torch.cat(scattered)
assert torch.allclose(recon, full_sum), "reduce-scatter values wrong"
print("✓ reduce-scatter: each rank holds 1/N of summed gradient")

# all-gather: reconstruct full tensor on every rank
shards    = [torch.arange(i * 10, (i + 1) * 10, dtype=torch.float) for i in range(N)]
gathered  = simulate_all_gather(shards)
full_ref  = torch.arange(N * 10, dtype=torch.float)
assert all(torch.allclose(g, full_ref) for g in gathered), "all-gather bug"
print("✓ all-gather: every rank has full reconstructed tensor")

# Communication volume comparison
elem_bytes = total_params * 4
print(f"\nCommunication volume (N={N}, {total_params} params × fp32):")
print(f"  all-reduce:      {2 * elem_bytes / 1024:.1f} KB  (2Ψ — DDP)")
print(f"  reduce-scatter:  {elem_bytes / 1024:.1f} KB  (Ψ  — ZeRO-2/3)")
print(f"  all-gather:      {elem_bytes / 1024:.1f} KB  (Ψ  — ZeRO-3 param recon)")

### 2c — Simulate a ZeRO-3 forward + backward step

Walk through exactly what happens per layer in ZeRO-3 / FSDP:

```
Forward pass (layer by layer):
  all-gather params for layer L  →  compute activations  →  free gathered params

Backward pass (reverse):
  all-gather params for layer L  →  compute gradients  →  reduce-scatter grads  →  free
```

In [ ]:
class ZeroThreeLayer:
    """
    Simulates a single linear layer under ZeRO-3 memory management.
    Each rank holds only a shard of the weight matrix.
    """
    def __init__(self, in_features: int, out_features: int, rank: int, world_size: int):
        self.rank       = rank
        self.world_size = world_size
        self.in_features  = in_features
        self.out_features = out_features

        # Full weight for reference (only used to create the shard)
        full_weight = torch.randn(out_features, in_features)

        # TODO: compute the shard this rank owns
        # Shard along the output (row) dimension
        # Each rank owns rows [rank * shard_rows : (rank+1) * shard_rows]
        self.shard_rows = out_features // world_size
        self.weight_shard = None  # shape: [shard_rows, in_features]
        raise NotImplementedError

        self.grad_shard = None   # filled during backward

    def all_gather_weights(self, all_shards: list) -> torch.Tensor:
        """
        Collect weight shards from all ranks and return the full weight matrix.
        This is what FSDP does at the start of each layer's forward pass.

        all_shards: list of weight_shard tensors from each rank

        TODO: implement using simulate_all_gather.
        Remember: flatten shards before gather, reshape after.
        """
        raise NotImplementedError

    def forward(self, x: torch.Tensor, all_weight_shards: list) -> torch.Tensor:
        """
        FSDP-style forward:
          1. All-gather to get full weight
          2. Compute linear transformation
          3. Free full weight (keep only shard)

        TODO: implement.
        Return (output, full_weight)  — caller needs full_weight for backward
        """
        raise NotImplementedError

    def backward(self, x: torch.Tensor, full_weight: torch.Tensor, grad_out: torch.Tensor, all_grad_shards: list):
        """
        FSDP-style backward:
          1. Compute full gradient w.r.t. weight: grad_w = grad_out.T @ x
          2. Compute gradient w.r.t. input: grad_x = grad_out @ full_weight
          3. Reduce-scatter grad_w — each rank accumulates its shard
          4. Free full grad_w

        all_grad_shards: pre-allocated list to put this rank's gradient shard into

        TODO: implement.
        Return grad_x so it can be passed to the previous layer.
        """
        raise NotImplementedError

In [ ]:
# Verify ZeroThreeLayer with a simple 2-rank simulation
N          = 2
in_f, out_f = 8, 4
batch_size = 3

layers = [ZeroThreeLayer(in_f, out_f, rank=r, world_size=N) for r in range(N)]

# Each rank processes the same input batch (simplified — real FSDP uses different batches)
x = torch.randn(batch_size, in_f)

# Collect shards from all ranks (simulating inter-GPU communication)
all_shards = [l.weight_shard for l in layers]

# Forward pass on rank 0
out, full_w = layers[0].forward(x, all_shards)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Full weight reconstructed: {full_w.shape}  (should be [{out_f}, {in_f}])")

# Backward
grad_out = torch.randn_like(out)
all_grad_shards = [None] * N
grad_x = layers[0].backward(x, full_w, grad_out, all_grad_shards)
print(f"grad_x shape: {grad_x.shape}  (should match input {x.shape})")

# Memory footprint after backward: each rank holds only 1/N of weight + grad
shard_bytes = layers[0].weight_shard.numel() * 4
full_bytes  = out_f * in_f * 4
print(f"\nMemory: shard={shard_bytes}B vs full={full_bytes}B  (ratio: {full_bytes/shard_bytes:.1f}x)")

---
## Stage 3 — DDP Baseline (Real Distributed Training)

Implement a real training loop with DDP using `torch.multiprocessing.spawn`.
This is the baseline to compare against FSDP.

> **Note**: If you only have 1 GPU or CPU, spawn with `world_size=1`.  
> DDP with 1 rank still exercises the init/cleanup code.

In [ ]:
def setup_process_group(rank: int, world_size: int, backend: str = "nccl"):
    """
    Initialize the distributed process group.
    Uses "gloo" backend on CPU, "nccl" on GPU.

    TODO: implement.
    Hint:
      os.environ['MASTER_ADDR'] = 'localhost'
      os.environ['MASTER_PORT'] = '12355'
      dist.init_process_group(backend, rank=rank, world_size=world_size)
    """
    raise NotImplementedError

def cleanup_process_group():
    """
    Destroy the process group after training.
    TODO: one line.
    """
    raise NotImplementedError

def get_batch(batch_size: int, seq_len: int, vocab_size: int, device):
    """Generate a random token batch for testing."""
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    return x, y

In [ ]:
# Worker functions are defined in fsdp_helpers.py so that mp.spawn can
# pickle them by module reference (notebook __main__ is not visible to
# spawned child processes).  See the module for full source code.
from fsdp_helpers import ddp_train_fn

In [ ]:
# Run DDP training
results_queue = mp.Queue()
mp.spawn(
    ddp_train_fn,
    args=(WORLD_SIZE, results_queue, USE_GPU, cfg),
    nprocs=WORLD_SIZE,
    join=True,
)
ddp_result = results_queue.get()
print("DDP result:", ddp_result)

---
## Stage 4 — FSDP (ZeRO-3) with PyTorch Native API

In [ ]:
from fsdp_helpers import fsdp_train_fn

In [ ]:
# Run FSDP with FULL_SHARD (ZeRO-3)
results_queue = mp.Queue()
mp.spawn(
    fsdp_train_fn,
    args=(WORLD_SIZE, results_queue, ShardingStrategy.FULL_SHARD, USE_GPU, cfg),
    nprocs=WORLD_SIZE,
    join=True,
)
fsdp_full_result = results_queue.get()
print("FSDP FULL_SHARD result:", fsdp_full_result)

# Run FSDP with SHARD_GRAD_OP (ZeRO-2)
results_queue = mp.Queue()
mp.spawn(
    fsdp_train_fn,
    args=(WORLD_SIZE, results_queue, ShardingStrategy.SHARD_GRAD_OP, USE_GPU, cfg),
    nprocs=WORLD_SIZE,
    join=True,
)
fsdp_grad_result = results_queue.get()
print("FSDP SHARD_GRAD_OP result:", fsdp_grad_result)

---
## Stage 5 — Memory & Communication Analysis

In [ ]:
# Combine all results and compare
all_results = [ddp_result, fsdp_full_result, fsdp_grad_result]

print(f"{'Method':<25}  {'Memory (MB)':>12}  {'Step time (ms)':>15}")
print("-" * 58)
for r in all_results:
    print(f"{r['method']:<25}  {r['peak_memory_mb']:>12.1f}  {r['mean_step_ms']:>15.1f}")

In [ ]:
def plot_memory_vs_communication(results_list: list, simulated_results: dict):
    """
    Two-panel plot:
    Left:  measured peak memory per method (bar chart)
    Right: theoretical memory scaling from Stage 2 simulated results
            (memory vs world_size for each ZeRO stage)

    TODO: implement both panels.
    """
    raise NotImplementedError

plot_memory_vs_communication(all_results, results)

In [ ]:
def communication_volume_analysis(n_params: int, world_size: int) -> dict:
    """
    Compute total bytes communicated per training step for each method.

    DDP:    all-reduce gradients = 2Ψ  (where Ψ = n_params × 4 bytes)
    ZeRO-1: reduce-scatter grads + all-gather param updates = 2Ψ  (same volume, different pattern)
    ZeRO-2: reduce-scatter grads only = Ψ
    ZeRO-3: reduce-scatter grads (Ψ) + 2 × all-gather params (2Ψ) = 3Ψ
            (all-gather needed twice per layer: once fwd, once bwd)

    Returns dict with keys: ddp, zero1, zero2, zero3  → bytes

    TODO: implement.
    Note: ZeRO-3 pays more in communication but saves memory.
    This is the fundamental tradeoff.
    """
    psi = n_params * 4  # bytes
    raise NotImplementedError

comm = communication_volume_analysis(n_params, WORLD_SIZE)
print("Communication volume per training step:")
for method, vol in comm.items():
    print(f"  {method:>8}: {vol / 1e9:.2f} GB")

---
## Extension Exercises

**1. Mixed precision FSDP**  
Wrap with `MixedPrecision(param_dtype=torch.bfloat16, reduce_dtype=torch.float32)`.  
Measure memory savings vs pure fp32. Re-run the memory breakdown analysis.

**2. Gradient checkpointing + FSDP**  
Apply `torch.utils.checkpoint` to each `TransformerBlock`.  
This trades activation memory for recomputation — measure the effect on peak memory.

**3. CPU offloading**  
Add `cpu_offload=CPUOffload(offload_params=True)` to FSDP.  
Observe: params live on CPU, all-gathered to GPU only when needed.  
Measure memory reduction vs throughput penalty.

**4. Implement ZeRO-1 optimizer**  
Write a custom optimizer wrapper that: (a) assigns each param to a rank,  
(b) only updates its own shard, (c) all-gathers updated params after step.  
Verify it matches a normal AdamW on a tiny model.

**5. Scale the model**  
Double `n_layers` and `d_model`. At what world_size does ZeRO-3 become necessary  
to fit on a single 40GB GPU? Use the Stage 2 simulation to find the crossover point.